# NLP Model Testing

Notebook ini menguji model koreksi teks dari `implementation.py` menggunakan metrik:
- ROUGE-1, ROUGE-2, ROUGE-L
- WER
- CER
- Accuracy (exact match)

Dataset default: `generated_kalimat.csv` dengan kolom `kalimat_awal` (target) dan `kalimat_salah` (input noise).

In [ ]:
# Jalankan sekali jika paket metrik belum terpasang
# %pip install -q pandas jiwer rouge-score

In [ ]:
import os
import re
import numpy as np
import pandas as pd
from jiwer import wer, cer
from rouge_score import rouge_scorer

from implementation import IndoBERTCorrector


def normalize_text(text: str) -> str:
    text = "" if text is None else str(text)
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text


def compute_rouge_average(references, predictions):
    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=False)
    agg = {"rouge1": [], "rouge2": [], "rougeL": []}

    for ref, pred in zip(references, predictions):
        scores = scorer.score(ref, pred)
        agg["rouge1"].append(scores["rouge1"].fmeasure)
        agg["rouge2"].append(scores["rouge2"].fmeasure)
        agg["rougeL"].append(scores["rougeL"].fmeasure)

    return {k: float(np.mean(v)) if v else 0.0 for k, v in agg.items()}

In [ ]:
# Konfigurasi data
DATA_PATH = "generated_kalimat.csv"  # ganti jika pakai file lain
MAX_SAMPLES = None  # isi angka (mis. 200) kalau ingin uji cepat

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"File dataset tidak ditemukan: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)
required_cols = {"kalimat_awal", "kalimat_salah"}
if not required_cols.issubset(df.columns):
    raise ValueError(f"Dataset wajib punya kolom: {required_cols}")

if MAX_SAMPLES is not None:
    df = df.head(int(MAX_SAMPLES)).copy()

# Normalisasi kolom untuk evaluasi yang konsisten
df["target"] = df["kalimat_awal"].map(normalize_text)
df["input_noisy"] = df["kalimat_salah"].map(normalize_text)

print(f"Jumlah sampel evaluasi: {len(df)}")

# Load model dari implementation.py
corrector = IndoBERTCorrector()
print("Model NLP berhasil dimuat.")

In [ ]:
# Inference model
preds = [normalize_text(corrector.correct(text)) for text in df["input_noisy"].tolist()]
refs = df["target"].tolist()
inputs = df["input_noisy"].tolist()

# Accuracy (exact match)
acc_model = float(np.mean([p == r for p, r in zip(preds, refs)]))
acc_noisy = float(np.mean([i == r for i, r in zip(inputs, refs)]))

# WER/CER
wer_model = float(wer(refs, preds))
cer_model = float(cer(refs, preds))
wer_noisy = float(wer(refs, inputs))
cer_noisy = float(cer(refs, inputs))

# ROUGE
rouge_model = compute_rouge_average(refs, preds)
rouge_noisy = compute_rouge_average(refs, inputs)

result_df = pd.DataFrame([
    {
        "setting": "noisy_input_baseline",
        "accuracy": acc_noisy,
        "wer": wer_noisy,
        "cer": cer_noisy,
        "rouge1": rouge_noisy["rouge1"],
        "rouge2": rouge_noisy["rouge2"],
        "rougeL": rouge_noisy["rougeL"],
    },
    {
        "setting": "indobert_corrected",
        "accuracy": acc_model,
        "wer": wer_model,
        "cer": cer_model,
        "rouge1": rouge_model["rouge1"],
        "rouge2": rouge_model["rouge2"],
        "rougeL": rouge_model["rougeL"],
    },
])

print("=== Evaluation Summary ===")
display(result_df)

preview = df[["kalimat_salah", "kalimat_awal"]].copy()
preview["prediksi_model"] = preds
preview = preview.rename(columns={"kalimat_salah": "input_noisy", "kalimat_awal": "target"})

print("\n=== Contoh Prediksi (20 baris pertama) ===")
display(preview.head(20))